In [1]:
!pip install -q transformers>=4.51.0 datasets torch accelerate sentencepiece protobuf


In [2]:
import torch
import json
import re
import os
import random
import time
from datetime import datetime
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Reproducibility
SEED = 42
random.seed(SEED)

# Config
MODEL_NAME = "Qwen/Qwen3-1.7B"
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.7
TOP_P = 0.8
TOP_K = 20
NUM_SAMPLES = 100

# Indic native-script configs (10 languages)
NATIVE_CONFIGS = ["bn", "gu", "hi", "kn", "ml", "mr", "or", "pa", "ta", "te"]
LANG_NAMES = {
    "bn": "Bengali", "gu": "Gujarati", "hi": "Hindi", "kn": "Kannada",
    "ml": "Malayalam", "mr": "Marathi", "or": "Odia", "pa": "Punjabi",
    "ta": "Tamil", "te": "Telugu"
}
# Romanized configs
ROMAN_CONFIGS = [f"{c}_roman" for c in NATIVE_CONFIGS]

# Samples per language (proportional: 100 / 10 = 10 each)
SAMPLES_PER_LANG = NUM_SAMPLES // len(NATIVE_CONFIGS)

# Results dir
os.makedirs("results", exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Samples per eval: {NUM_SAMPLES}")
print(f"Samples per language: {SAMPLES_PER_LANG}")
print(f"Native languages: {NATIVE_CONFIGS}")
print(f"Romanized languages: {ROMAN_CONFIGS}")


Model: Qwen/Qwen3-1.7B
Samples per eval: 100
Samples per language: 10
Native languages: ['bn', 'gu', 'hi', 'kn', 'ml', 'mr', 'or', 'pa', 'ta', 'te']
Romanized languages: ['bn_roman', 'gu_roman', 'hi_roman', 'kn_roman', 'ml_roman', 'mr_roman', 'or_roman', 'pa_roman', 'ta_roman', 'te_roman']


In [3]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print(f"Model loaded on {model.device}")
print(f"Model dtype: {model.dtype}")


Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded on cuda:0
Model dtype: torch.bfloat16


In [ ]:
# --- English GSM8K ---
print("Loading English GSM8K...")
gsm8k_en = load_dataset("openai/gsm8k", "main", split="test")
en_indices = random.sample(range(len(gsm8k_en)), NUM_SAMPLES)
en_samples = gsm8k_en.select(en_indices)
print(f"  English GSM8K: {len(en_samples)} samples selected from {len(gsm8k_en)} total")

# --- Indic Native Script ---
print("\nLoading Indic GSM8K (native script)...")
native_samples = []
for cfg in NATIVE_CONFIGS:
    ds = load_dataset("sarvamai/gsm8k-indic", cfg, split="test")
    indices = random.sample(range(len(ds)), SAMPLES_PER_LANG)
    for idx in indices:
        row = ds[idx]
        native_samples.append({
            "lang": cfg,
            "lang_name": LANG_NAMES[cfg],
            "question": row["question"],
            "answer": row["answer"],
            "original_question": row.get("original_question", ""),
            "original_answer": row.get("original_answer", ""),
        })
    print(f"  {LANG_NAMES[cfg]} ({cfg}): {SAMPLES_PER_LANG} samples from {len(ds)} total")
print(f"  Total native samples: {len(native_samples)}")

# --- Indic Romanized ---
print("\nLoading Indic GSM8K (romanized)...")
roman_samples = []
for cfg in ROMAN_CONFIGS:
    ds = load_dataset("sarvamai/gsm8k-indic", cfg, split="test")
    indices = random.sample(range(len(ds)), SAMPLES_PER_LANG)
    base_lang = cfg.replace("_roman", "")
    for idx in indices:
        row = ds[idx]
        roman_samples.append({
            "lang": cfg,
            "lang_name": f"{LANG_NAMES[base_lang]} (Roman)",
            "question": row["question"],
            "answer": row["answer"],
            "original_question": row.get("original_question", ""),
            "original_answer": row.get("original_answer", ""),
        })
    print(f"  {LANG_NAMES[base_lang]} romanized ({cfg}): {SAMPLES_PER_LANG} samples from {len(ds)} total")
print(f"  Total romanized samples: {len(roman_samples)}")

print(f"\n=== TOTAL: {len(en_samples)} EN + {len(native_samples)} native + {len(roman_samples)} roman = {len(en_samples)+len(native_samples)+len(roman_samples)} evals ===")


Loading English GSM8K...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/1303 [00:00<?, ? examples/s]

  Tamil romanized (ta_roman): 10 samples from 1303 total


te_roman/test-00000-of-00001.parquet:   0%|          | 0.00/628k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

  Telugu romanized (te_roman): 10 samples from 1319 total
  Total romanized samples: 100

=== TOTAL: 100 EN + 100 native + 100 roman = 300 evals ===


In [4]:
def extract_gold_answer(answer_str):
    """Extract numeric answer from GSM8K format: '... #### <number>'"""
    match = re.search(r'####\s*(-?[\d,]+\.?\d*)', answer_str)
    if match:
        return match.group(1).replace(",", "").strip()
    return None

def extract_model_answer(text):
    """Extract the last numeric value from model output as the predicted answer."""
    # Look for boxed answer first (model might use \boxed{})
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        num = re.sub(r'[^\d.\-]', '', boxed[-1])
        if num:
            return num

    # Look for "answer is X" or "= X" patterns
    patterns = [
        r'(?:answer|result|total|solution)\s*(?:is|=|:)\s*(-?[\d,]+\.?\d*)',
        r'####\s*(-?[\d,]+\.?\d*)',
        r'=\s*(-?[\d,]+\.?\d*)\s*$',
    ]
    for pat in patterns:
        matches = re.findall(pat, text, re.IGNORECASE | re.MULTILINE)
        if matches:
            return matches[-1].replace(",", "").strip()

    # Fallback: last number in text
    numbers = re.findall(r'-?[\d,]+\.?\d*', text)
    if numbers:
        return numbers[-1].replace(",", "").strip()

    return None

def compare_answers(pred, gold):
    """Compare predicted and gold answers with tolerance."""
    if pred is None or gold is None:
        return False
    try:
        p = float(pred)
        g = float(gold)
        return abs(p - g) < 1e-3
    except (ValueError, TypeError):
        return pred.strip() == gold.strip()

def generate_answer(question, lang_hint=""):
    """Generate model answer for a math question."""
    if lang_hint:
        prompt = f"Solve this math problem. Show your work step by step and give the final numeric answer.\n\nQuestion: {question}"
    else:
        prompt = f"Solve this math problem. Show your work step by step and give the final numeric answer.\n\nQuestion: {question}"

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            do_sample=True,
        )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    return response

print("Helper functions defined.")
# Quick test of answer extraction
assert extract_gold_answer("Some work #### 72") == "72"
assert extract_gold_answer("Some work #### 1,234") == "1234"
print("Answer extraction tests passed.")


Helper functions defined.
Answer extraction tests passed.


In [ ]:
print("=" * 80)
print("EVAL 1/3: English GSM8K (100 samples)")
print("=" * 80)

en_logs = []
en_correct = 0
start_time = time.time()

for i in range(len(en_samples)):
    question = en_samples[i]["question"]
    gold_answer_raw = en_samples[i]["answer"]
    gold = extract_gold_answer(gold_answer_raw)

    print(f"\n--- [{i+1}/{NUM_SAMPLES}] ---")
    print(f"Q: {question[:120]}...")

    response = generate_answer(question)
    pred = extract_model_answer(response)
    correct = compare_answers(pred, gold)

    if correct:
        en_correct += 1

    log_entry = {
        "index": i,
        "question": question,
        "gold_answer_raw": gold_answer_raw,
        "gold_answer": gold,
        "model_response": response,
        "predicted_answer": pred,
        "correct": correct,
    }
    en_logs.append(log_entry)

    print(f"Gold: {gold} | Pred: {pred} | {'✓' if correct else '✗'}")
    print(f"Running accuracy: {en_correct}/{i+1} = {en_correct/(i+1)*100:.1f}%")

en_elapsed = time.time() - start_time
en_accuracy = en_correct / NUM_SAMPLES * 100

print(f"\n{'=' * 80}")
print(f"ENGLISH GSM8K RESULTS: {en_correct}/{NUM_SAMPLES} = {en_accuracy:.1f}%")
print(f"Time: {en_elapsed:.1f}s ({en_elapsed/NUM_SAMPLES:.1f}s per sample)")
print(f"{'=' * 80}")


EVAL 1/3: English GSM8K (100 samples)

--- [1/100] ---
Q: The girls are trying to raise money for a carnival. Kim raises $320 more than Alexandra, who raises $430, and Maryam rai...
Gold: 2280 | Pred: 2180 | ✗
Running accuracy: 0/1 = 0.0%

--- [2/100] ---
Q: Kalinda is working on a 360 piece puzzle with her mom. Kalinda can normally add 4 pieces per minute. Her mom can typical...
Gold: 1 | Pred: 1 | ✓
Running accuracy: 1/2 = 50.0%

--- [3/100] ---
Q: Tom's ship can travel at 10 miles per hour.  He is sailing from 1 to 4 PM.  He then travels back at a rate of 6 mph.  Ho...
Gold: 5 | Pred: 5 | ✓
Running accuracy: 2/3 = 66.7%

--- [4/100] ---
Q: James decides to buy birthday candles for his 2 sons.  One of them is 12 and the other is 4 years younger.  A pack of 5 ...
Gold: 12 | Pred: 12 | ✓
Running accuracy: 3/4 = 75.0%

--- [5/100] ---
Q: Mariah’s grandma was teaching her to knit. Mariah used 1/4 of a skein of yarn. Her grandma used 1/2 of a skein of yarn. ...
Gold: 273 | Pred: 273 | ✓
R

In [ ]:
print("=" * 80)
print("EVAL 2/3: Indic GSM8K — Native Script (100 samples, 10 per language)")
print("=" * 80)

native_logs = []
native_correct = 0
native_per_lang = {cfg: {"correct": 0, "total": 0} for cfg in NATIVE_CONFIGS}
start_time = time.time()

for i, sample in enumerate(native_samples):
    lang = sample["lang"]
    question = sample["question"]
    gold_answer_raw = sample["answer"]
    gold = extract_gold_answer(gold_answer_raw)

    # If gold extraction fails on translated answer, try original_answer
    if gold is None and sample["original_answer"]:
        gold = extract_gold_answer(sample["original_answer"])

    print(f"\n--- [{i+1}/{len(native_samples)}] [{sample['lang_name']}] ---")
    print(f"Q: {question[:120]}...")

    response = generate_answer(question, lang_hint=sample["lang_name"])
    pred = extract_model_answer(response)
    correct = compare_answers(pred, gold)

    if correct:
        native_correct += 1
        native_per_lang[lang]["correct"] += 1
    native_per_lang[lang]["total"] += 1

    log_entry = {
        "index": i,
        "lang": lang,
        "lang_name": sample["lang_name"],
        "question": question,
        "original_question": sample["original_question"],
        "gold_answer_raw": gold_answer_raw,
        "gold_answer": gold,
        "model_response": response,
        "predicted_answer": pred,
        "correct": correct,
    }
    native_logs.append(log_entry)

    print(f"Gold: {gold} | Pred: {pred} | {'✓' if correct else '✗'}")
    print(f"Running accuracy: {native_correct}/{i+1} = {native_correct/(i+1)*100:.1f}%")

native_elapsed = time.time() - start_time
native_accuracy = native_correct / len(native_samples) * 100

print(f"\n{'=' * 80}")
print(f"INDIC NATIVE SCRIPT RESULTS: {native_correct}/{len(native_samples)} = {native_accuracy:.1f}%")
print(f"Time: {native_elapsed:.1f}s ({native_elapsed/len(native_samples):.1f}s per sample)")
print(f"\nPer-language breakdown:")
for cfg in NATIVE_CONFIGS:
    stats = native_per_lang[cfg]
    acc = stats['correct'] / stats['total'] * 100 if stats['total'] > 0 else 0
    print(f"  {LANG_NAMES[cfg]:12s} ({cfg}): {stats['correct']}/{stats['total']} = {acc:.0f}%")
print(f"{'=' * 80}")


EVAL 2/3: Indic GSM8K — Native Script (100 samples, 10 per language)

--- [1/100] [Bengali] ---
Q: জন 10টি স্কচ বোতল কেনেন যার মোট মূল্য $600। তিনি আরও দ্বিগুণ সংখ্যক কনিয়াক বোতল কেনেন যার প্রতি বোতলের দাম 50% বেশি। তি...
Gold: 2400 | Pred: 2400 | ✓
Running accuracy: 1/1 = 100.0%

--- [2/100] [Bengali] ---
Q: $2000 লাভ করতে, ইসাইয়াসকে তার খামার থেকে বাজারে নিয়ে যাওয়ার জন্য পরিকল্পিত মুরগিগুলো প্রতিটি $50 দরে বিক্রি করতে হবে।...
Gold: 7000 | Pred: 9000 | ✗
Running accuracy: 1/2 = 50.0%

--- [3/100] [Bengali] ---
Q: মিস্টার ম্যাক্সিম দ্য বেস্ট কুকারিজ অ্যারাউন্ড রেস্টুরেন্টে কাজ করেন। একটি নির্দিষ্ট দিনে, সকালে 50 জন লোক খাওয়ার জন্য ...
Gold: 320 | Pred: 280 | ✗
Running accuracy: 1/3 = 33.3%

--- [4/100] [Bengali] ---
Q: পল শনিবারে একটি জন্মদিনের পার্টির জন্য 63টি কাপকেক প্রয়োজন। তার ইতিমধ্যে 8টি চকোলেট কাপকেক এবং 40টি টফি কাপকেক আছে। পলক...
Gold: 15 | Pred: 15 | ✓
Running accuracy: 2/4 = 50.0%

--- [5/100] [Bengali] ---
Q: Jeremy তাদের বাড়ির পিছনে 12টি পাখি দেখল এবং তাদের দিকে এক

In [ ]:
print("=" * 80)
print("EVAL 3/3: Indic GSM8K — Romanized (100 samples, 10 per language)")
print("=" * 80)

roman_logs = []
roman_correct = 0
roman_per_lang = {cfg: {"correct": 0, "total": 0} for cfg in ROMAN_CONFIGS}
start_time = time.time()

for i, sample in enumerate(roman_samples):
    lang = sample["lang"]
    question = sample["question"]
    gold_answer_raw = sample["answer"]
    gold = extract_gold_answer(gold_answer_raw)

    # If gold extraction fails on translated answer, try original_answer
    if gold is None and sample["original_answer"]:
        gold = extract_gold_answer(sample["original_answer"])

    print(f"\n--- [{i+1}/{len(roman_samples)}] [{sample['lang_name']}] ---")
    print(f"Q: {question[:120]}...")

    response = generate_answer(question, lang_hint=sample["lang_name"])
    pred = extract_model_answer(response)
    correct = compare_answers(pred, gold)

    if correct:
        roman_correct += 1
        roman_per_lang[lang]["correct"] += 1
    roman_per_lang[lang]["total"] += 1

    log_entry = {
        "index": i,
        "lang": lang,
        "lang_name": sample["lang_name"],
        "question": question,
        "original_question": sample["original_question"],
        "gold_answer_raw": gold_answer_raw,
        "gold_answer": gold,
        "model_response": response,
        "predicted_answer": pred,
        "correct": correct,
    }
    roman_logs.append(log_entry)

    print(f"Gold: {gold} | Pred: {pred} | {'✓' if correct else '✗'}")
    print(f"Running accuracy: {roman_correct}/{i+1} = {roman_correct/(i+1)*100:.1f}%")

roman_elapsed = time.time() - start_time
roman_accuracy = roman_correct / len(roman_samples) * 100

print(f"\n{'=' * 80}")
print(f"INDIC ROMANIZED RESULTS: {roman_correct}/{len(roman_samples)} = {roman_accuracy:.1f}%")
print(f"Time: {roman_elapsed:.1f}s ({roman_elapsed/len(roman_samples):.1f}s per sample)")
print(f"\nPer-language breakdown:")
for cfg in ROMAN_CONFIGS:
    stats = roman_per_lang[cfg]
    base = cfg.replace("_roman", "")
    acc = stats['correct'] / stats['total'] * 100 if stats['total'] > 0 else 0
    print(f"  {LANG_NAMES[base]:12s} ({cfg}): {stats['correct']}/{stats['total']} = {acc:.0f}%")
print(f"{'=' * 80}")


EVAL 3/3: Indic GSM8K — Romanized (100 samples, 10 per language)

--- [1/100] [Bengali (Roman)] ---
Q: Jackie gormokale 3" berechhe. Se ekhon Aner-er cheye 2" chhoto, jini Albert-er aakarer digun. Jodi Albert-er uchchota 36...
Gold: 67 | Pred: 3 | ✗
Running accuracy: 0/1 = 0.0%

--- [2/100] [Bengali (Roman)] ---
Q: Miguel tar akar jonyo proti soptahe 2ti pad kagaj byabohar kore. Jodi ekti pad kagaje 30 sheet kagaj thake, tahole se pr...
Gold: 240 | Pred: 30 | ✗
Running accuracy: 0/2 = 0.0%

--- [3/100] [Bengali (Roman)] ---
Q: Fireman Franker-er 200 juta achhe. Jodi se sombar-e 5 jora juta pae ebong budhbar-e 15 notun jora ebong shukrobar-e 30 j...
Gold: 120 | Pred: 831000000 | ✗
Running accuracy: 0/3 = 0.0%

--- [4/100] [Bengali (Roman)] ---
Q: Mark goto kal ekti porikkha diyechhilo jate 75ti proshno chhilo. Se proti ghontae 5ti proshner hare porikkhati somponno ...
Gold: 105 | Pred: 180 | ✗
Running accuracy: 0/4 = 0.0%

--- [5/100] [Bengali (Roman)] ---
Q: Sofia tar garite ekti road 

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# --- Build summary ---
summary = {
    "model": MODEL_NAME,
    "timestamp": timestamp,
    "seed": SEED,
    "num_samples_per_eval": NUM_SAMPLES,
    "generation_config": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "enable_thinking": False,
    },
    "results": {
        "english_gsm8k": {
            "correct": en_correct,
            "total": NUM_SAMPLES,
            "accuracy": round(en_accuracy, 2),
            "time_seconds": round(en_elapsed, 1),
        },
        "indic_native": {
            "correct": native_correct,
            "total": len(native_samples),
            "accuracy": round(native_accuracy, 2),
            "time_seconds": round(native_elapsed, 1),
            "per_language": {
                cfg: {
                    "lang_name": LANG_NAMES[cfg],
                    "correct": native_per_lang[cfg]["correct"],
                    "total": native_per_lang[cfg]["total"],
                    "accuracy": round(native_per_lang[cfg]["correct"] / native_per_lang[cfg]["total"] * 100, 2) if native_per_lang[cfg]["total"] > 0 else 0,
                }
                for cfg in NATIVE_CONFIGS
            },
        },
        "indic_romanized": {
            "correct": roman_correct,
            "total": len(roman_samples),
            "accuracy": round(roman_accuracy, 2),
            "time_seconds": round(roman_elapsed, 1),
            "per_language": {
                cfg: {
                    "lang_name": LANG_NAMES[cfg.replace("_roman", "")],
                    "correct": roman_per_lang[cfg]["correct"],
                    "total": roman_per_lang[cfg]["total"],
                    "accuracy": round(roman_per_lang[cfg]["correct"] / roman_per_lang[cfg]["total"] * 100, 2) if roman_per_lang[cfg]["total"] > 0 else 0,
                }
                for cfg in ROMAN_CONFIGS
            },
        },
    },
}

# --- Save all logs ---
all_logs = {
    "summary": summary,
    "english_logs": en_logs,
    "native_logs": native_logs,
    "romanized_logs": roman_logs,
}

logs_path = f"results/eval_logs_{timestamp}.json"
summary_path = f"results/eval_summary_{timestamp}.json"

with open(logs_path, "w", encoding="utf-8") as f:
    json.dump(all_logs, f, ensure_ascii=False, indent=2)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Logs saved to: {logs_path}")
print(f"Summary saved to: {summary_path}")

# --- Print final summary table ---
print(f"\n{'=' * 80}")
print(f"FINAL RESULTS — {MODEL_NAME}")
print(f"{'=' * 80}")
print(f"{'Benchmark':<30} {'Correct':>8} {'Total':>6} {'Accuracy':>10} {'Time':>8}")
print(f"{'-'*30} {'-'*8} {'-'*6} {'-'*10} {'-'*8}")
print(f"{'English GSM8K':<30} {en_correct:>8} {NUM_SAMPLES:>6} {en_accuracy:>9.1f}% {en_elapsed:>7.0f}s")
print(f"{'Indic Native Script':<30} {native_correct:>8} {len(native_samples):>6} {native_accuracy:>9.1f}% {native_elapsed:>7.0f}s")
print(f"{'Indic Romanized':<30} {roman_correct:>8} {len(roman_samples):>6} {roman_accuracy:>9.1f}% {roman_elapsed:>7.0f}s")
print(f"{'-'*30} {'-'*8} {'-'*6} {'-'*10} {'-'*8}")
total_c = en_correct + native_correct + roman_correct
total_n = NUM_SAMPLES + len(native_samples) + len(roman_samples)
print(f"{'OVERALL':<30} {total_c:>8} {total_n:>6} {total_c/total_n*100:>9.1f}%")

print(f"\n--- Indic Native Per-Language ---")
for cfg in NATIVE_CONFIGS:
    s = native_per_lang[cfg]
    a = s['correct']/s['total']*100 if s['total']>0 else 0
    print(f"  {LANG_NAMES[cfg]:12s}: {s['correct']}/{s['total']} = {a:5.1f}%")

print(f"\n--- Indic Romanized Per-Language ---")
for cfg in ROMAN_CONFIGS:
    s = roman_per_lang[cfg]
    b = cfg.replace("_roman","")
    a = s['correct']/s['total']*100 if s['total']>0 else 0
    print(f"  {LANG_NAMES[b]:12s}: {s['correct']}/{s['total']} = {a:5.1f}%")


Logs saved to: results/eval_logs_20260415_132252.json
Summary saved to: results/eval_summary_20260415_132252.json

FINAL RESULTS — Qwen/Qwen3-1.7B
Benchmark                       Correct  Total   Accuracy     Time
------------------------------ -------- ------ ---------- --------
English GSM8K                        86    100      86.0%    1280s
Indic Native Script                  41    100      41.0%    1678s
Indic Romanized                       7    100       7.0%    2699s
------------------------------ -------- ------ ---------- --------
OVERALL                             134    300      44.7%

--- Indic Native Per-Language ---
  Bengali     : 5/10 =  50.0%
  Gujarati    : 5/10 =  50.0%
  Hindi       : 5/10 =  50.0%
  Kannada     : 5/10 =  50.0%
  Malayalam   : 3/10 =  30.0%
  Marathi     : 2/10 =  20.0%
  Odia        : 4/10 =  40.0%
  Punjabi     : 4/10 =  40.0%
  Tamil       : 3/10 =  30.0%
  Telugu      : 5/10 =  50.0%

--- Indic Romanized Per-Language ---
  Bengali     : 1/10

In [ ]:
# ============================================================
# EVALUATION - Native Script Punjabi Only (100 Samples)
# ============================================================
import time
from datasets import load_dataset
import random

print("=" * 80)
print("EVAL: Indic GSM8K — Native Script Punjabi (100 samples)")
print("=" * 80)

print("Loading Indic GSM8K (pa)...")
# Load the Punjabi native split
pa_ds = load_dataset("sarvamai/gsm8k-indic", "pa", split="test")

# Sample exactly 100 indices for evaluation
NUM_EVAL_SAMPLES = 100
indices = random.sample(range(len(pa_ds)), min(NUM_EVAL_SAMPLES, len(pa_ds)))

pa_samples = []
for idx in indices:
    row = pa_ds[idx]
    pa_samples.append({
        "lang_name": "Punjabi",
        "question": row["question"],
        "answer": row["answer"],
        "original_answer": row.get("original_answer", ""),
    })

print(f"Total Punjabi samples selected: {len(pa_samples)}")

pa_logs = []
pa_correct = 0
start_time = time.time()

for i, sample in enumerate(pa_samples):
    question = sample["question"]
    gold_answer_raw = sample["answer"]
    
    # Extract gold answer using the helper function defined in the notebook
    gold = extract_gold_answer(gold_answer_raw)

    # Fallback to English answer if translated answer doesn't have obvious gold numeric value
    if gold is None and sample["original_answer"]:
        gold = extract_gold_answer(sample["original_answer"])

    print(f"\n--- [{i+1}/{len(pa_samples)}] [Punjabi] ---")
    print(f"Q: {question[:120]}...")

    # Generate model response using your helper
    response = generate_answer(question, lang_hint="Punjabi")
    pred = extract_model_answer(response)
    correct = compare_answers(pred, gold)

    if correct:
        pa_correct += 1

    log_entry = {
        "index": i,
        "question": question,
        "gold_answer": gold,
        "predicted_answer": pred,
        "correct": correct,
    }
    pa_logs.append(log_entry)

    print(f"Gold: {str(gold):>6s} | Pred: {str(pred):>6s} | {'✓' if correct else '✗'}")
    print(f"Running accuracy: {pa_correct}/{i+1} = {pa_correct/(i+1)*100:.1f}%")

pa_elapsed = time.time() - start_time
pa_accuracy = pa_correct / len(pa_samples) * 100

# Print final results
print(f"\n{'=' * 80}")
print(f"PUNJABI NATIVE SCRIPT RESULTS: {pa_correct}/{len(pa_samples)} = {pa_accuracy:.1f}%")
print(f"Time: {pa_elapsed:.1f}s ({pa_elapsed/len(pa_samples):.1f}s per sample)")
print(f"{'=' * 80}")


EVAL: Indic GSM8K — Native Script Punjabi (100 samples)
Loading Indic GSM8K (pa)...


README.md: 0.00B [00:00, ?B/s]

pa/test-00000-of-00001.parquet:   0%|          | 0.00/711k [00:00<?, ?B/s]